# Intervention Effectiveness Pipeline

This notebook builds an **intervention effectiveness pipeline** for the INTEX nonprofit project.

It is designed to improve on the earlier notebooks by doing four things more explicitly:

- include a true **business framing**
- include both an **explanatory / relationship model** and a **predictive model**
- include stronger **deployment-ready outputs**
- keep the notebook organized like a real **end-to-end pipeline**

The notebook follows the chapter flow you provided:

- **Chapter 1** for CRISP-DM framing and business fit
- **Chapter 6** for reusable, automated, and error-resistant EDA patterns
- **Chapter 7** for dataset-specific wrangling followed by reusable preparation logic
- **Chapters 11, 13, 14, 15, and 16** for predictive modeling, classification, ensemble comparison, cross-validation, and safe feature selection
- **Chapter 17** for deployment-minded outputs and model artifacts

## 1. Problem Framing

### Business question

Which intervention plans appear to be associated with better near-term resident outcomes, and which active resident-months are most likely to result in a **successful next-month intervention outcome**?

### Why this matters

The organization needs more than notes and intuition. Staff need a practical way to:

- see which intervention patterns are linked with improvement
- identify residents who may need escalation or closer monitoring
- prioritize scarce staff time
- feed meaningful analytics into the website and internal dashboard

### Two-model design

This notebook intentionally separates two goals:

1. **Explanatory / relationship analysis**  
   Estimate which intervention characteristics are associated with next-month change in resident wellbeing, while controlling for baseline condition and recent case activity.

2. **Predictive modeling**  
   Predict whether a resident-month with active interventions is likely to produce a successful next-month outcome.

This split matches the course distinction between **explanation** and **prediction** and improves alignment with the IS 455 requirements.

## 2. Data Acquisition, Preparation & Exploration

This section:

- locates the INTEX CSV files using flexible file discovery
- standardizes column names and common aliases
- builds a resident-month intervention panel
- creates lagged and baseline controls
- defines both a continuous explanatory outcome and a binary predictive outcome
- saves the modeling panel for review

### Notebook setup and reusable helper functions

This block keeps the notebook robust for VS Code reruns and for different local folder layouts.

Outputs are saved to:

`ml-pipelines/generated_outputs`

when the notebook is run from the project root or from inside the `ml-pipelines` folder.

In [2]:
# Core imports and notebook configuration.
import json
import math
import os
import re
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")

# Make display output easier to read in the notebook.
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

# Reproducibility setting.
SEED = 42

# Resolve the intended ml-pipelines project root.
def resolve_project_root() -> Path:
    current = Path.cwd()

    if current.name == "ml-pipelines":
        return current

    direct_child = current / "ml-pipelines"
    if direct_child.exists():
        return direct_child

    for parent in [current] + list(current.parents):
        if parent.name == "ml-pipelines":
            return parent
        candidate = parent / "ml-pipelines"
        if candidate.exists():
            return candidate

    return current

PROJECT_ROOT = resolve_project_root()
LOCAL_OUTPUT_DIR = PROJECT_ROOT / "generated_outputs"
OUTPUT_DIR = Path(os.environ.get("AZUREML_OUTPUT_DIR", str(LOCAL_OUTPUT_DIR)))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

Project root: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines
Output directory: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs


In [3]:
# Helper functions for robust file discovery and standardization.

def normalize_name(value: str) -> str:
    """Return a simplified lowercase name for flexible file and column matching."""
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def candidate_roots() -> list[Path]:
    """Search common local locations for CSV tables without hard-coding one folder."""
    roots = [
        PROJECT_ROOT.parent / "data",
        PROJECT_ROOT / "data",
        Path("../data"),
        Path("./data"),
        Path("/mnt/batch/tasks/shared/LS_root/mounts/clusters/notebookdev/code/data"),
        Path("/lighthouse/lighthouse_csv_v7/lighthouse_csv_v7"),
        Path("/lighthouse/lighthouse_csv_v7"),
        Path("/lighthouse"),
        PROJECT_ROOT,
        PROJECT_ROOT / "Data",
        PROJECT_ROOT / "csv",
        PROJECT_ROOT / "CSV",
        PROJECT_ROOT / "files",
        PROJECT_ROOT / "dataset",
        PROJECT_ROOT / "datasets",
        PROJECT_ROOT.parent,
        PROJECT_ROOT.parent / "datasets",
    ]

    unique_roots = []
    seen = set()

    for root in roots:
        if root.exists():
            resolved = root.resolve()
            if resolved not in seen:
                unique_roots.append(root)
                seen.add(resolved)

    return unique_roots


def discover_csv_files(search_roots: list[Path]) -> dict[str, Path]:
    """Recursively discover CSV files and index them by normalized stem and filename."""
    found = {}

    for root in search_roots:
        for path in root.rglob("*.csv"):
            stem_key = normalize_name(path.stem)
            file_key = normalize_name(path.name)
            found.setdefault(stem_key, path)
            found.setdefault(file_key, path)

    return found


def resolve_table_path(alias_options: list[str], file_index: dict[str, Path]) -> Path | None:
    """Return the first matching path from a list of possible table names."""
    for alias in alias_options:
        key = normalize_name(alias)
        if key in file_index:
            return file_index[key]
    return None


def safe_read_csv(path: Path) -> pd.DataFrame:
    """Read a CSV with a short encoding fallback list."""
    encodings = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception as exc:
            last_error = exc

    raise last_error


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize column names and trim string values."""
    df = df.copy()
    df.columns = [normalize_name(col) for col in df.columns]

    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"": np.nan, "nan": np.nan, "None": np.nan, "none": np.nan})

    return df


def first_existing(df: pd.DataFrame, options: list[str]) -> str | None:
    """Return the first column that exists from a list of aliases."""
    normalized_map = {normalize_name(col): col for col in df.columns}

    for option in options:
        key = normalize_name(option)
        if key in normalized_map:
            return normalized_map[key]

    return None


def rename_first_match(df: pd.DataFrame, alias_map: dict[str, list[str]]) -> pd.DataFrame:
    """Rename the first found alias in each alias list to the target column name."""
    df = df.copy()
    rename_dict = {}

    for target_name, options in alias_map.items():
        match = first_existing(df, options)
        if match is not None and match != target_name:
            rename_dict[match] = target_name

    if rename_dict:
        df = df.rename(columns=rename_dict)

    return df


def month_floor(series: pd.Series) -> pd.Series:
    """Convert a datetime-like series to first-of-month timestamps."""
    return pd.to_datetime(series, errors="coerce").dt.to_period("M").dt.to_timestamp()


def rate_from_series(series: pd.Series) -> float:
    """Return a safe mean for binary or rate-like features."""
    if series.dropna().empty:
        return np.nan
    return float(series.mean())


SEARCH_ROOTS = candidate_roots()
FILE_INDEX = discover_csv_files(SEARCH_ROOTS)

print("Search roots checked:")
for root in SEARCH_ROOTS:
    print(" -", root)

print(f"Discovered CSV entries: {len(FILE_INDEX)}")

Search roots checked:
 - c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines
 - c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026
Discovered CSV entries: 46


### File discovery and table loading

This notebook expects the intervention and case-management tables to be available as CSV files.

The notebook is intentionally flexible about exact filenames and common schema variations.

In [4]:
# Define likely INTEX table aliases and load available tables.

TABLE_ALIASES = {
    "residents": ["residents", "resident", "resident_master"],
    "intervention_plans": ["intervention_plans", "intervention_plan", "plans"],
    "health_records": ["health_wellbeing_records", "health_records", "health_record", "health", "wellbeing_records", "wellbeing"],
    "incident_reports": ["incident_reports", "incident_report", "incidents", "incident"],
    "process_recordings": ["process_recordings", "process_recording", "counseling_sessions", "process_sessions"],
    "safehouses": ["safehouses", "safehouse", "homes"],
}

tables = {}
table_paths = {}

for logical_name, aliases in TABLE_ALIASES.items():
    path = resolve_table_path(aliases, FILE_INDEX)
    table_paths[logical_name] = path

    if path is not None:
        df = safe_read_csv(path)
        df = standardize_columns(df)
        tables[logical_name] = df
        print(f"Loaded {logical_name:<20} -> {path.name:<35} shape={df.shape}")
    else:
        tables[logical_name] = pd.DataFrame()
        print(f"Missing {logical_name:<20} -> no matching CSV found")

required_tables = ["intervention_plans", "health_records"]
missing_required = [name for name in required_tables if tables[name].empty]

if missing_required:
    raise FileNotFoundError(
        f"This notebook requires the following source tables: {missing_required}. "
        "Please make sure those CSV files are available in or under the ml-pipelines folder."
    )

Loaded residents            -> residents.csv                       shape=(60, 49)
Loaded intervention_plans   -> intervention_plans.csv              shape=(180, 11)
Loaded health_records       -> health_wellbeing_records.csv        shape=(534, 14)
Loaded incident_reports     -> incident_reports.csv                shape=(100, 12)
Loaded process_recordings   -> process_recordings.csv              shape=(2819, 15)
Loaded safehouses           -> safehouses.csv                      shape=(9, 13)


### Initial table review

A quick structural review helps confirm which tables are present and what each table contributes to the modeling panel.

In [5]:
# Review the loaded tables.

table_review = pd.DataFrame(
    [
        {
            "table_name": name,
            "loaded": not df.empty,
            "rows": int(df.shape[0]),
            "columns": int(df.shape[1]),
            "path": str(table_paths.get(name)) if table_paths.get(name) is not None else None,
        }
        for name, df in tables.items()
    ]
)

display(table_review)

,table_name,loaded,rows,columns,path
0,residents,True,60,49,c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml...
1,intervention_plans,True,180,11,c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml...
2,health_records,True,534,14,c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml...
3,incident_reports,True,100,12,c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml...
4,process_recordings,True,2819,15,c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml...
5,safehouses,True,9,13,c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml...


### Column standardization

This block creates stable names such as `resident_id`, `safehouse_id`, `plan_date`, and `health_score` even when the raw source tables use slightly different column names.

In [6]:
# Standardize key columns across tables.

COMMON_ALIAS_MAP = {
    "resident_id": ["resident_id", "client_id", "girl_id", "beneficiary_id", "case_id"],
    "safehouse_id": ["safehouse_id", "home_id", "shelter_id"],
    "created_date": ["created_date", "created_at", "create_date"],
    "record_date": ["record_date", "date"],
}

PLAN_ALIAS_MAP = {
    "resident_id": ["resident_id", "client_id", "girl_id", "beneficiary_id", "case_id"],
    "safehouse_id": ["safehouse_id", "home_id", "shelter_id"],
    "plan_date": ["plan_date", "record_date", "date", "created_date"],
    "plan_status": ["plan_status", "status"],
    "plan_type": ["plan_type", "type", "intervention_type"],
    "goal_area": ["goal_area", "focus_area", "program_area"],
}

HEALTH_ALIAS_MAP = {
    "resident_id": ["resident_id", "client_id", "girl_id", "beneficiary_id", "case_id"],
    "safehouse_id": ["safehouse_id", "home_id", "shelter_id"],
    "health_date": ["health_date", "record_date", "date"],
    "health_score": ["health_score", "overall_health_score", "wellbeing_score", "wellbeing_index"],
    "sleep_quality_score": ["sleep_quality_score", "sleep_score"],
    "energy_level_score": ["energy_level_score", "energy_score"],
}

INCIDENT_ALIAS_MAP = {
    "resident_id": ["resident_id", "client_id", "girl_id", "beneficiary_id", "case_id"],
    "safehouse_id": ["safehouse_id", "home_id", "shelter_id"],
    "incident_date": ["incident_date", "record_date", "date"],
    "severity": ["severity", "incident_severity", "severity_level"],
    "follow_up_required": ["follow_up_required", "requires_follow_up", "follow_up"],
}

PROCESS_ALIAS_MAP = {
    "resident_id": ["resident_id", "client_id", "girl_id", "beneficiary_id", "case_id"],
    "safehouse_id": ["safehouse_id", "home_id", "shelter_id"],
    "session_date": ["session_date", "record_date", "date", "process_recording_date"],
    "session_type": ["session_type", "type", "counseling_type"],
    "session_duration_minutes": ["session_duration_minutes", "duration_minutes", "minutes", "session_length_minutes"],
}

RESIDENT_ALIAS_MAP = {
    "resident_id": ["resident_id", "client_id", "girl_id", "beneficiary_id", "case_id"],
    "safehouse_id": ["safehouse_id", "home_id", "shelter_id"],
    "date_of_birth": ["date_of_birth", "dob", "birth_date"],
    "case_category": ["case_category", "primary_case_category", "category"],
    "case_status": ["case_status", "status", "resident_status"],
}

SAFEHOUSE_ALIAS_MAP = {
    "safehouse_id": ["safehouse_id", "home_id", "shelter_id"],
    "safehouse_name": ["safehouse_name", "name", "home_name"],
    "region": ["region", "area"],
    "open_date": ["open_date", "start_date", "opened_date"],
}

for table_name, df in list(tables.items()):
    df = rename_first_match(df, COMMON_ALIAS_MAP)

    if table_name == "intervention_plans":
        df = rename_first_match(df, PLAN_ALIAS_MAP)
    elif table_name == "health_records":
        df = rename_first_match(df, HEALTH_ALIAS_MAP)
    elif table_name == "incident_reports":
        df = rename_first_match(df, INCIDENT_ALIAS_MAP)
    elif table_name == "process_recordings":
        df = rename_first_match(df, PROCESS_ALIAS_MAP)
    elif table_name == "residents":
        df = rename_first_match(df, RESIDENT_ALIAS_MAP)
    elif table_name == "safehouses":
        df = rename_first_match(df, SAFEHOUSE_ALIAS_MAP)

    tables[table_name] = df

# Convert likely date columns.
date_candidates = [
    "plan_date",
    "health_date",
    "incident_date",
    "session_date",
    "created_date",
    "record_date",
    "date_of_birth",
    "open_date",
]

for table_name, df in list(tables.items()):
    for date_col in date_candidates:
        if date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    tables[table_name] = df

for name, df in tables.items():
    print(f"\n{name}")
    print(df.columns.tolist())


residents
['resident_id', 'case_control_no', 'internal_code', 'safehouse_id', 'case_status', 'sex', 'date_of_birth', 'birth_status', 'place_of_birth', 'religion', 'case_category', 'sub_cat_orphaned', 'sub_cat_trafficked', 'sub_cat_child_labor', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse', 'sub_cat_osaec', 'sub_cat_cicl', 'sub_cat_at_risk', 'sub_cat_street_child', 'sub_cat_child_with_hiv', 'is_pwd', 'pwd_type', 'has_special_needs', 'special_needs_diagnosis', 'family_is_4ps', 'family_solo_parent', 'family_indigenous', 'family_parent_pwd', 'family_informal_settler', 'date_of_admission', 'age_upon_admission', 'present_age', 'length_of_stay', 'referral_source', 'referring_agency_person', 'date_colb_registered', 'date_colb_obtained', 'assigned_social_worker', 'initial_case_assessment', 'date_case_study_prepared', 'reintegration_type', 'reintegration_status', 'initial_risk_level', 'current_risk_level', 'date_enrolled', 'date_closed', 'created_date', 'notes_restricted']

intervention_pla

### Resident master enrichment

This cell standardizes resident-level context fields that may help both the explanatory and predictive sections.

In [7]:
# Clean and enrich the resident master table.

residents = tables["residents"].copy()

if not residents.empty:
    if "date_of_birth" in residents.columns:
        today_ref = pd.Timestamp.today().normalize()
        residents["resident_age_years"] = ((today_ref - residents["date_of_birth"]).dt.days / 365.25).round(1)

    for col in ["resident_age_years"]:
        if col in residents.columns:
            residents[col] = pd.to_numeric(residents[col], errors="coerce")

    resident_keep_cols = [
        col for col in [
            "resident_id",
            "safehouse_id",
            "resident_age_years",
            "case_category",
            "case_status",
        ] if col in residents.columns
    ]
    residents = residents[resident_keep_cols].drop_duplicates(subset=["resident_id"]).copy()

display(residents.head())

,resident_id,safehouse_id,resident_age_years,case_category,case_status
0,1,4,17.6,Neglected,Active
1,2,3,18.0,Surrendered,Closed
2,3,1,19.2,Surrendered,Active
3,4,2,13.8,Neglected,Active
4,5,4,17.0,Surrendered,Transferred


### Build the resident-month intervention base table

This table is the core unit of analysis. Each row represents a **resident-month with at least one intervention plan**.

In [8]:
# Build the resident-month intervention base table.

plans = tables["intervention_plans"].copy()

if "plan_date" not in plans.columns:
    if "created_date" in plans.columns:
        plans["plan_date"] = plans["created_date"]

plans["metric_month"] = month_floor(plans["plan_date"])
plans = plans.dropna(subset=["resident_id", "metric_month"]).copy()

for col in ["plan_status", "plan_type", "goal_area"]:
    if col in plans.columns:
        plans[col] = plans[col].astype(str).str.strip().str.lower()

# Create useful binary indicators for plan status and common intervention categories.
if "plan_status" in plans.columns:
    plans["plan_achieved_flag"] = plans["plan_status"].isin(["achieved", "completed", "resolved", "done"]).astype(int)
    plans["plan_active_flag"] = plans["plan_status"].isin(["active", "ongoing", "open", "in_progress"]).astype(int)
else:
    plans["plan_achieved_flag"] = 0
    plans["plan_active_flag"] = 0

if "plan_type" in plans.columns:
    plans["health_plan_flag"] = plans["plan_type"].str.contains("health|medical|wellbeing", na=False).astype(int)
    plans["education_plan_flag"] = plans["plan_type"].str.contains("education|school|academic", na=False).astype(int)
    plans["safety_plan_flag"] = plans["plan_type"].str.contains("safety|protection|reintegration|family", na=False).astype(int)
else:
    plans["health_plan_flag"] = 0
    plans["education_plan_flag"] = 0
    plans["safety_plan_flag"] = 0

plan_group_cols = ["resident_id", "metric_month"]
if "safehouse_id" in plans.columns:
    plan_group_cols.append("safehouse_id")

intervention_month = (
    plans
    .groupby(plan_group_cols, as_index=False)
    .agg(
        plans_created=("resident_id", "size"),
        achieved_plans=("plan_achieved_flag", "sum"),
        active_plans=("plan_active_flag", "sum"),
        health_plans=("health_plan_flag", "sum"),
        education_plans=("education_plan_flag", "sum"),
        safety_plans=("safety_plan_flag", "sum"),
    )
)

intervention_month["achieved_plan_share"] = intervention_month["achieved_plans"] / intervention_month["plans_created"].replace(0, np.nan)
intervention_month["health_plan_share"] = intervention_month["health_plans"] / intervention_month["plans_created"].replace(0, np.nan)
intervention_month["education_plan_share"] = intervention_month["education_plans"] / intervention_month["plans_created"].replace(0, np.nan)
intervention_month["safety_plan_share"] = intervention_month["safety_plans"] / intervention_month["plans_created"].replace(0, np.nan)

display(intervention_month.head())

,resident_id,metric_month,plans_created,achieved_plans,active_plans,health_plans,education_plans,safety_plans,achieved_plan_share,health_plan_share,education_plan_share,safety_plan_share
0,1,2023-10-01,3,0,0,0,0,0,0.000000,0.0,0.0,0.0
1,2,2023-03-01,3,1,1,0,0,0,0.333333,0.0,0.0,0.0
2,3,2024-05-01,3,0,2,0,0,0,0.000000,0.0,0.0,0.0
3,4,2024-09-01,3,0,1,0,0,0,0.000000,0.0,0.0,0.0
4,5,2024-01-01,3,1,0,0,0,0,0.333333,0.0,0.0,0.0


### Supporting monthly outcome and activity tables

The next cells summarize:

- health and wellbeing records
- incident activity
- counseling / process recording activity

In [9]:
# Build the monthly health table.

health = tables["health_records"].copy()
health["metric_month"] = month_floor(health["health_date"])
health = health.dropna(subset=["resident_id", "metric_month"]).copy()

for col in ["health_score", "sleep_quality_score", "energy_level_score"]:
    if col in health.columns:
        health[col] = pd.to_numeric(health[col], errors="coerce")

health_group_cols = ["resident_id", "metric_month"]
if "safehouse_id" in health.columns:
    health_group_cols.append("safehouse_id")

health_agg_map = {}
for col in ["health_score", "sleep_quality_score", "energy_level_score"]:
    if col in health.columns:
        health_agg_map[col] = "mean"

health_month = health.groupby(health_group_cols, as_index=False).agg(health_agg_map)
health_counts = (
    health.groupby(health_group_cols, as_index=False)
    .size()
    .rename(columns={"size": "health_record_count"})
)

health_month = health_month.merge(health_counts, on=health_group_cols, how="left")

# Create a composite wellbeing score when enough health fields exist.
composite_inputs = [col for col in ["health_score", "sleep_quality_score", "energy_level_score"] if col in health_month.columns]
if composite_inputs:
    health_month["wellbeing_composite"] = health_month[composite_inputs].mean(axis=1)

display(health_month.head())

,resident_id,metric_month,sleep_quality_score,energy_level_score,health_record_count,wellbeing_composite
0,1,2023-10-01,3.18,2.90,1,3.040
1,1,2023-11-01,3.18,2.85,1,3.015
2,1,2023-12-01,3.19,2.94,1,3.065
3,1,2024-01-01,3.21,2.92,1,3.065
4,1,2024-02-01,3.26,2.93,1,3.095


In [10]:
# Build the monthly incident table.

incidents = tables["incident_reports"].copy()

if not incidents.empty and "incident_date" in incidents.columns:
    incidents["metric_month"] = month_floor(incidents["incident_date"])
    incidents = incidents.dropna(subset=["resident_id", "metric_month"]).copy()

    if "severity" in incidents.columns:
        incidents["severity_clean"] = incidents["severity"].astype(str).str.strip().str.lower()
        incidents["high_severity_incident_flag"] = incidents["severity_clean"].isin(["high", "severe", "critical"]).astype(int)
    else:
        incidents["high_severity_incident_flag"] = 0

    incident_group_cols = ["resident_id", "metric_month"]
    if "safehouse_id" in incidents.columns:
        incident_group_cols.append("safehouse_id")

    incident_month = (
        incidents
        .groupby(incident_group_cols, as_index=False)
        .agg(
            incident_count=("resident_id", "size"),
            high_severity_incidents=("high_severity_incident_flag", "sum"),
        )
    )
else:
    incident_month = pd.DataFrame(columns=["resident_id", "metric_month", "incident_count", "high_severity_incidents"])

display(incident_month.head())

,resident_id,metric_month,safehouse_id,incident_count,high_severity_incidents
0,1,2024-02-01,4,1,0
1,1,2024-06-01,4,1,0
2,1,2025-04-01,4,1,0
3,1,2026-02-01,4,1,1
4,3,2025-02-01,1,1,0


In [11]:
# Build the monthly process-recording table.

process = tables["process_recordings"].copy()

if not process.empty and "session_date" in process.columns:
    process["metric_month"] = month_floor(process["session_date"])
    process = process.dropna(subset=["resident_id", "metric_month"]).copy()

    if "session_duration_minutes" in process.columns:
        process["session_duration_minutes"] = pd.to_numeric(process["session_duration_minutes"], errors="coerce")

    process_group_cols = ["resident_id", "metric_month"]
    if "safehouse_id" in process.columns:
        process_group_cols.append("safehouse_id")

    agg_map = {"resident_id": "size"}
    rename_map = {"resident_id": "process_session_count"}

    if "session_duration_minutes" in process.columns:
        agg_map["session_duration_minutes"] = "mean"
        rename_map["session_duration_minutes"] = "avg_session_duration"

    process_month = process.groupby(process_group_cols, as_index=False).agg(agg_map)
    process_month = process_month.rename(columns=rename_map)
else:
    process_month = pd.DataFrame(columns=["resident_id", "metric_month", "process_session_count", "avg_session_duration"])

display(process_month.head())

,metric_month,process_session_count,avg_session_duration
0,2023-11-01,3,74.000000
1,2023-12-01,6,75.333333
2,2024-01-01,3,60.666667
3,2024-02-01,3,44.333333
4,2024-03-01,5,71.400000


### Merge all monthly features into one intervention panel

This panel uses the intervention month as the base and layers in resident, safehouse, health, incident, and process features.

In [12]:
# Merge all monthly features into one panel.
# This version is more defensive than the first draft:
# it builds the base resident-month panel from whatever monthly tables actually have rows,
# instead of assuming intervention_month is always populated.

monthly_sources = []
source_names = []

for name, source in [
    ("intervention_month", intervention_month),
    ("health_month", health_month),
    ("incident_month", incident_month),
    ("process_month", process_month),
]:
    if source is not None and not source.empty:
        key_cols = [col for col in ["resident_id", "metric_month", "safehouse_id"] if col in source.columns]
        if key_cols:
            monthly_sources.append(source[key_cols].drop_duplicates().copy())
            source_names.append(name)

if monthly_sources:
    # Union all available resident-month keys so the panel still exists
    # even if one source table is empty.
    panel = pd.concat(monthly_sources, ignore_index=True).drop_duplicates().copy()
else:
    # Create an empty shell with expected keys so downstream code can fail gracefully.
    panel = pd.DataFrame(columns=["resident_id", "metric_month", "safehouse_id"])

print("Monthly sources contributing to the panel base:", source_names)
print("Base panel key shape before merges:", panel.shape)

# Merge the feature tables back onto the shared resident-month key frame.
for source in [intervention_month, health_month, incident_month, process_month]:
    if source is None or source.empty:
        continue

    join_keys = [col for col in ["resident_id", "metric_month", "safehouse_id"] if col in panel.columns and col in source.columns]

    if not join_keys:
        join_keys = [col for col in ["resident_id", "metric_month"] if col in panel.columns and col in source.columns]

    if join_keys:
        panel = panel.merge(source, on=join_keys, how="left")

# Add resident-level attributes for controls and segmentation.
if not residents.empty and "resident_id" in panel.columns and "resident_id" in residents.columns:
    resident_non_overlaps = [col for col in residents.columns if col not in panel.columns or col == "resident_id"]
    panel = panel.merge(residents[resident_non_overlaps], on="resident_id", how="left")

# Add safehouse metadata for reporting and modeling.
safehouses = tables["safehouses"].copy()
if not safehouses.empty and "safehouse_id" in panel.columns and "safehouse_id" in safehouses.columns:
    safehouse_keep_cols = [col for col in ["safehouse_id", "safehouse_name", "region", "open_date"] if col in safehouses.columns]
    safehouse_lookup = safehouses[safehouse_keep_cols].drop_duplicates(subset=["safehouse_id"]).copy()
    panel = panel.merge(safehouse_lookup, on="safehouse_id", how="left")

display(panel.head())
print("Panel shape before target creation:", panel.shape)

Monthly sources contributing to the panel base: ['intervention_month', 'health_month', 'incident_month', 'process_month']
Base panel key shape before merges: (677, 3)


,resident_id,metric_month,safehouse_id,plans_created,achieved_plans,active_plans,health_plans,education_plans,safety_plans,achieved_plan_share,health_plan_share,education_plan_share,safety_plan_share,sleep_quality_score,energy_level_score,health_record_count,wellbeing_composite,incident_count,high_severity_incidents,process_session_count,avg_session_duration,resident_age_years,case_category,case_status,safehouse_name,region,open_date
0,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,NaN,NaN,4,70.250000,17.6,Neglected,Active,NaN,NaN,NaT
1,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,NaN,NaN,6,62.166667,17.6,Neglected,Active,NaN,NaN,NaT
2,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,NaN,NaN,1,105.000000,17.6,Neglected,Active,NaN,NaN,NaT
3,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,NaN,NaN,3,63.000000,17.6,Neglected,Active,NaN,NaN,NaT
4,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,NaN,NaN,2,79.000000,17.6,Neglected,Active,NaN,NaN,NaT


Panel shape before target creation: (18922, 27)


### Create future outcomes carefully

The notebook creates both:

- a **continuous explanatory outcome**: next-month change in wellbeing composite
- a **binary predictive target**: intervention success next month

The target uses only information that would occur **after** the current intervention month.

In [13]:
# Create forward-looking outcomes.
# This section builds the next-period target in a way that tolerates imperfect time coverage.
# It prefers a true next-month comparison when month information exists, but it can also
# fall back to the next observed resident record when the source data is sparse.

# Always sort before using grouped lead logic.
sort_cols = [col for col in ["resident_id", "metric_month"] if col in panel.columns]
if sort_cols:
    panel = panel.sort_values(sort_cols).reset_index(drop=True)

# Forward-looking values from the next observed resident record.
if "metric_month" in panel.columns:
    panel["next_metric_month"] = panel.groupby("resident_id")["metric_month"].shift(-1)
else:
    panel["next_metric_month"] = pd.NaT

if "wellbeing_composite" in panel.columns:
    panel["next_month_wellbeing_composite"] = panel.groupby("resident_id")["wellbeing_composite"].shift(-1)
else:
    panel["next_month_wellbeing_composite"] = np.nan

if "incident_count" in panel.columns:
    panel["next_month_incident_count"] = panel.groupby("resident_id")["incident_count"].shift(-1)
else:
    panel["next_month_incident_count"] = np.nan

if "high_severity_incidents" in panel.columns:
    panel["next_month_high_severity_incidents"] = panel.groupby("resident_id")["high_severity_incidents"].shift(-1)
else:
    panel["next_month_high_severity_incidents"] = np.nan

# Determine whether the next observed record is truly the next month.
if "metric_month" in panel.columns and panel["metric_month"].notna().sum() > 0:
    expected_next_month = panel["metric_month"] + pd.offsets.MonthBegin(1)
    panel["has_true_next_month"] = panel["next_metric_month"].eq(expected_next_month)
else:
    panel["has_true_next_month"] = False

# Build a continuous change target.
# Prefer a strict month-to-month change when that information exists.
# If not enough strict matches exist, use the next observed resident record as the change comparison.
strict_change = np.where(
    panel["has_true_next_month"],
    panel["next_month_wellbeing_composite"] - panel["wellbeing_composite"],
    np.nan,
)

fallback_change = panel["next_month_wellbeing_composite"] - panel["wellbeing_composite"]
panel["next_month_wellbeing_change"] = strict_change

if pd.Series(panel["next_month_wellbeing_change"]).notna().sum() == 0:
    panel["next_month_wellbeing_change"] = fallback_change

# Define a binary success label.
# A successful next-period outcome means wellbeing improves and no high-severity incident is recorded
# in the next available observation window.
panel["intervention_success_next_month"] = np.where(
    panel["next_month_wellbeing_change"].notna(),
    np.where(
        (panel["next_month_wellbeing_change"] > 0)
        & panel["next_month_high_severity_incidents"].fillna(0).eq(0),
        1,
        0,
    ),
    np.nan,
)

display(
    panel[
        [
            col for col in [
                "resident_id",
                "metric_month",
                "wellbeing_composite",
                "next_month_wellbeing_composite",
                "next_month_wellbeing_change",
                "intervention_success_next_month",
            ] if col in panel.columns
        ]
    ].head(12)
)

print("Rows with non-null continuous target:", panel["next_month_wellbeing_change"].notna().sum())
print("Rows with non-null binary target:", pd.Series(panel["intervention_success_next_month"]).notna().sum())

,resident_id,metric_month,wellbeing_composite,next_month_wellbeing_composite,next_month_wellbeing_change,intervention_success_next_month
0,1.0,2023-10-01,3.04,3.04,NaN,NaN
1,1.0,2023-10-01,3.04,3.04,NaN,NaN
2,1.0,2023-10-01,3.04,3.04,NaN,NaN
3,1.0,2023-10-01,3.04,3.04,NaN,NaN
4,1.0,2023-10-01,3.04,3.04,NaN,NaN
5,1.0,2023-10-01,3.04,3.04,NaN,NaN
6,1.0,2023-10-01,3.04,3.04,NaN,NaN
7,1.0,2023-10-01,3.04,3.04,NaN,NaN
8,1.0,2023-10-01,3.04,3.04,NaN,NaN
9,1.0,2023-10-01,3.04,3.04,NaN,NaN


Rows with non-null continuous target: 474
Rows with non-null binary target: 474


### Lagged, rolling, and calendar features

This step gives the predictive model richer baseline context while still respecting time order.

In [14]:
# Add lagged, rolling, and calendar features.
# These features let the predictive model use prior resident trajectory information
# without leaking future outcomes.

panel = panel.sort_values(["resident_id", "metric_month"]).reset_index(drop=True)

lag_candidates = [
    col for col in [
        "wellbeing_composite",
        "health_score",
        "sleep_quality_score",
        "energy_level_score",
        "plans_created",
        "achieved_plans",
        "active_plans",
        "health_plans",
        "education_plans",
        "safety_plans",
        "incident_count",
        "high_severity_incidents",
        "process_session_count",
        "avg_session_duration",
    ] if col in panel.columns
]

for col in lag_candidates:
    # Lag features capture the recent past without peeking at the future.
    panel[f"{col}_lag1"] = panel.groupby("resident_id")[col].shift(1)
    panel[f"{col}_lag2"] = panel.groupby("resident_id")[col].shift(2)

rolling_candidates = [col for col in ["wellbeing_composite", "incident_count", "process_session_count"] if col in panel.columns]
for col in rolling_candidates:
    # Rolling means summarize short-run trajectory instead of one-month noise.
    panel[f"{col}_roll3_mean"] = (
        panel.groupby("resident_id")[col]
        .transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean())
    )

# Calendar features are helpful when real month values exist.
if "metric_month" in panel.columns and panel["metric_month"].notna().sum() > 0:
    panel["month_num"] = panel["metric_month"].dt.month
    panel["year_num"] = panel["metric_month"].dt.year
else:
    panel["month_num"] = np.nan
    panel["year_num"] = np.nan

# A resident-relative sequence index still works even when absolute calendar values are sparse.
panel["resident_month_index"] = panel.groupby("resident_id").cumcount() + 1

if "open_date" in panel.columns and "metric_month" in panel.columns and panel["metric_month"].notna().sum() > 0:
    panel["safehouse_age_days"] = (panel["metric_month"] - panel["open_date"]).dt.days

display(panel.head())

,resident_id,metric_month,safehouse_id,plans_created,achieved_plans,active_plans,health_plans,education_plans,safety_plans,achieved_plan_share,health_plan_share,education_plan_share,safety_plan_share,sleep_quality_score,energy_level_score,health_record_count,wellbeing_composite,incident_count,high_severity_incidents,process_session_count,avg_session_duration,resident_age_years,case_category,case_status,safehouse_name,region,open_date,next_metric_month,next_month_wellbeing_composite,next_month_incident_count,next_month_high_severity_incidents,has_true_next_month,next_month_wellbeing_change,intervention_success_next_month,wellbeing_composite_lag1,wellbeing_composite_lag2,sleep_quality_score_lag1,sleep_quality_score_lag2,energy_level_score_lag1,energy_level_score_lag2,plans_created_lag1,plans_created_lag2,achieved_plans_lag1,achieved_plans_lag2,active_plans_lag1,active_plans_lag2,health_plans_lag1,health_plans_lag2,education_plans_lag1,education_plans_lag2,safety_plans_lag1,safety_plans_lag2,incident_count_lag1,incident_count_lag2,high_severity_incidents_lag1,high_severity_incidents_lag2,process_session_count_lag1,process_session_count_lag2,avg_session_duration_lag1,avg_session_duration_lag2,wellbeing_composite_roll3_mean,incident_count_roll3_mean,process_session_count_roll3_mean,month_num,year_num,resident_month_index,safehouse_age_days
0,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,NaN,NaN,4,70.250000,17.6,Neglected,Active,NaN,NaN,NaT,2023-10-01,3.04,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,2023,1.0,NaN
1,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,NaN,NaN,6,62.166667,17.6,Neglected,Active,NaN,NaN,NaT,2023-10-01,3.04,NaN,NaN,False,NaN,NaN,3.04,NaN,3.18,NaN,2.9,NaN,3.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,4.0,NaN,70.250000,NaN,3.04,NaN,4.000000,10,2023,2.0,NaN
2,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,NaN,NaN,1,105.000000,17.6,Neglected,Active,NaN,NaN,NaT,2023-10-01,3.04,NaN,NaN,False,NaN,NaN,3.04,3.04,3.18,3.18,2.9,2.9,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,6.0,4.0,62.166667,70.250000,3.04,NaN,5.000000,10,2023,3.0,NaN
3,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,NaN,NaN,3,63.000000,17.6,Neglected,Active,NaN,NaN,NaT,2023-10-01,3.04,NaN,NaN,False,NaN,NaN,3.04,3.04,3.18,3.18,2.9,2.9,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,1.0,6.0,105.000000,62.166667,3.04,NaN,3.666667,10,2023,4.0,NaN
4,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,NaN,NaN,2,79.000000,17.6,Neglected,Active,NaN,NaN,NaT,2023-10-01,3.04,NaN,NaN,False,NaN,NaN,3.04,3.04,3.18,3.18,2.9,2.9,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,3.0,1.0,63.000000,105.000000,3.04,NaN,3.333333,10,2023,5.0,NaN


### Final panel cleanup before modeling

This step keeps only rows with a valid next-month label and saves the full panel for auditability.

In [15]:
# Final panel cleanup.
# This version keeps rows whenever a usable target can be built.
# The earlier version dropped all rows because it required two fields
# that were not simultaneously available in the user's data.

panel_model = panel.copy()

# Fill count-like and share-like features with zero when missing.
zero_fill_keywords = [
    "count",
    "plans",
    "incidents",
    "session_count",
    "record_count",
    "flag",
    "share",
]
zero_fill_cols = [col for col in panel_model.columns if any(keyword in col for keyword in zero_fill_keywords)]

for col in zero_fill_cols:
    if col in panel_model.columns:
        panel_model[col] = panel_model[col].fillna(0)

# Build or repair the binary target if needed.
if "intervention_success_next_month" not in panel_model.columns:
    panel_model["intervention_success_next_month"] = np.nan

# First choice: use the continuous next-period change if it exists.
if "next_month_wellbeing_change" in panel_model.columns and panel_model["next_month_wellbeing_change"].notna().sum() > 0:
    panel_model["intervention_success_next_month"] = (
        panel_model["next_month_wellbeing_change"] > 0
    ).astype(float)

# Second choice: derive the change directly from current and next observed wellbeing.
elif (
    "next_month_wellbeing_composite" in panel_model.columns
    and "wellbeing_composite" in panel_model.columns
):
    panel_model["next_month_wellbeing_change"] = (
        panel_model["next_month_wellbeing_composite"] - panel_model["wellbeing_composite"]
    )
    panel_model["intervention_success_next_month"] = (
        panel_model["next_month_wellbeing_change"] > 0
    ).astype(float)

# Keep only rows with a usable target.
panel_model = panel_model[panel_model["intervention_success_next_month"].notna()].copy()
panel_model["intervention_success_next_month"] = panel_model["intervention_success_next_month"].astype(int)

panel_output_path = OUTPUT_DIR / "intervention_effectiveness_panel.csv"
panel_model.to_csv(panel_output_path, index=False)

print("Modeling panel shape:", panel_model.shape)
print("Panel saved to:", panel_output_path)

if "intervention_success_next_month" in panel_model.columns:
    print("Target counts:")
    print(panel_model["intervention_success_next_month"].value_counts(dropna=False))

display(panel_model.head())

Modeling panel shape: (18922, 67)
Panel saved to: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs\intervention_effectiveness_panel.csv
Target counts:
intervention_success_next_month
0    18605
1      317
Name: count, dtype: int64


,resident_id,metric_month,safehouse_id,plans_created,achieved_plans,active_plans,health_plans,education_plans,safety_plans,achieved_plan_share,health_plan_share,education_plan_share,safety_plan_share,sleep_quality_score,energy_level_score,health_record_count,wellbeing_composite,incident_count,high_severity_incidents,process_session_count,avg_session_duration,resident_age_years,case_category,case_status,safehouse_name,region,open_date,next_metric_month,next_month_wellbeing_composite,next_month_incident_count,next_month_high_severity_incidents,has_true_next_month,next_month_wellbeing_change,intervention_success_next_month,wellbeing_composite_lag1,wellbeing_composite_lag2,sleep_quality_score_lag1,sleep_quality_score_lag2,energy_level_score_lag1,energy_level_score_lag2,plans_created_lag1,plans_created_lag2,achieved_plans_lag1,achieved_plans_lag2,active_plans_lag1,active_plans_lag2,health_plans_lag1,health_plans_lag2,education_plans_lag1,education_plans_lag2,safety_plans_lag1,safety_plans_lag2,incident_count_lag1,incident_count_lag2,high_severity_incidents_lag1,high_severity_incidents_lag2,process_session_count_lag1,process_session_count_lag2,avg_session_duration_lag1,avg_session_duration_lag2,wellbeing_composite_roll3_mean,incident_count_roll3_mean,process_session_count_roll3_mean,month_num,year_num,resident_month_index,safehouse_age_days
0,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,0.0,0.0,4,70.250000,17.6,Neglected,Active,NaN,NaN,NaT,2023-10-01,3.04,0.0,0.0,False,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.000000,10,2023,1.0,NaN
1,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,0.0,0.0,6,62.166667,17.6,Neglected,Active,NaN,NaN,NaT,2023-10-01,3.04,0.0,0.0,False,NaN,0,3.04,NaN,3.18,NaN,2.9,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,70.250000,NaN,3.04,0.0,4.000000,10,2023,2.0,NaN
2,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,0.0,0.0,1,105.000000,17.6,Neglected,Active,NaN,NaN,NaT,2023-10-01,3.04,0.0,0.0,False,NaN,0,3.04,3.04,3.18,3.18,2.9,2.9,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,4.0,62.166667,70.250000,3.04,0.0,5.000000,10,2023,3.0,NaN
3,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,0.0,0.0,3,63.000000,17.6,Neglected,Active,NaN,NaN,NaT,2023-10-01,3.04,0.0,0.0,False,NaN,0,3.04,3.04,3.18,3.18,2.9,2.9,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,6.0,105.000000,62.166667,3.04,0.0,3.666667,10,2023,4.0,NaN
4,1.0,2023-10-01,NaN,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.18,2.9,1.0,3.04,0.0,0.0,2,79.000000,17.6,Neglected,Active,NaN,NaN,NaT,2023-10-01,3.04,0.0,0.0,False,NaN,0,3.04,3.04,3.18,3.18,2.9,2.9,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,63.000000,105.000000,3.04,0.0,3.333333,10,2023,5.0,NaN


### Exploratory checks

A few quick checks help verify class balance, month coverage, and the size of the final intervention panel.

In [16]:
# Exploratory checks for balance and time coverage.

print("Success rate:")
print(panel_model["intervention_success_next_month"].value_counts(dropna=False))
print()
print("Months covered:")
print(panel_model["metric_month"].min(), "to", panel_model["metric_month"].max())
print()
print("Residents covered:", panel_model["resident_id"].nunique())

month_summary = (
    panel_model.groupby("metric_month", as_index=False)
    .agg(
        resident_months=("resident_id", "size"),
        success_rate=("intervention_success_next_month", "mean"),
        avg_wellbeing=("wellbeing_composite", "mean"),
    )
)

display(month_summary.head(12))

Success rate:
intervention_success_next_month
0    18605
1      317
Name: count, dtype: int64

Months covered:
2023-01-01 00:00:00 to 2027-02-01 00:00:00

Residents covered: 60


,metric_month,resident_months,success_rate,avg_wellbeing
0,2023-01-01,2,0.500000,2.955000
1,2023-02-01,5,0.800000,2.970000
2,2023-03-01,48,0.104167,3.000714
3,2023-04-01,96,0.072917,2.955000
4,2023-05-01,96,0.093750,3.003182
5,2023-06-01,176,0.034091,3.036667
6,2023-07-01,234,0.055556,3.033824
7,2023-08-01,285,0.045614,3.045000
8,2023-09-01,361,0.030471,3.036111
9,2023-10-01,462,0.030303,3.054762


## 3. Explanatory / Relationship Analysis

This section is intentionally separate from the predictive model.

It focuses on a continuous outcome:

`next_month_wellbeing_change`

The goal is not to maximize classification accuracy. The goal is to estimate whether certain intervention patterns are associated with more positive next-month change after controlling for baseline condition and recent activity.

This is the notebook section meant to improve the weaker explanatory coverage in the earlier pipelines.

### Build the explanatory modeling table

The explanatory model uses a more selective feature set with a focus on interpretability.

In [17]:
# Build a smaller explanatory modeling table.
# The explanatory section is intentionally simpler and more interpretable than the predictive section.
# It focuses on which intervention mix and resident-state variables are associated with next-period change.

explanatory_base_cols = [
    "next_month_wellbeing_change",
    "wellbeing_composite",
    "plans_created",
    "achieved_plan_share",
    "health_plan_share",
    "education_plan_share",
    "safety_plan_share",
    "incident_count",
    "high_severity_incidents",
    "process_session_count",
    "avg_session_duration",
    "resident_age_years",
    "resident_month_index",
    "month_num",
    "case_category",
    "case_status",
    "region",
]

explanatory_cols = [col for col in explanatory_base_cols if col in panel_model.columns]
explanatory_df = panel_model[explanatory_cols].copy()

# Keep only rows with the continuous explanatory target.
if "next_month_wellbeing_change" in explanatory_df.columns:
    explanatory_df = explanatory_df.dropna(subset=["next_month_wellbeing_change"]).copy()

# Fill numeric controls with medians so OLS does not drop rows unnecessarily.
for col in explanatory_df.select_dtypes(include=["number"]).columns:
    if col != "next_month_wellbeing_change":
        explanatory_df[col] = explanatory_df[col].fillna(explanatory_df[col].median())

# Fill categorical controls with an explicit unknown bucket.
for col in explanatory_df.select_dtypes(include=["object"]).columns:
    explanatory_df[col] = explanatory_df[col].fillna("unknown")

display(explanatory_df.head())
print("Explanatory table shape:", explanatory_df.shape)

,next_month_wellbeing_change,wellbeing_composite,plans_created,achieved_plan_share,health_plan_share,education_plan_share,safety_plan_share,incident_count,high_severity_incidents,process_session_count,avg_session_duration,resident_age_years,resident_month_index,month_num,case_category,case_status,region
20,-0.025,3.040,3.0,0.0,0.0,0.0,0.0,0.0,0.0,6,72.833333,17.6,21.0,10,Neglected,Active,unknown
43,0.050,3.015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,58.000000,17.6,44.0,11,Neglected,Active,unknown
66,0.000,3.065,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,47.000000,17.6,67.0,12,Neglected,Active,unknown
94,0.030,3.065,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,61.000000,17.6,95.0,1,Neglected,Active,unknown
152,-0.040,3.095,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,35.000000,17.6,153.0,2,Neglected,Active,Visayas


Explanatory table shape: (474, 17)


### Dummy coding and VIF review

This follows the course guidance to treat explanatory feature selection differently from predictive feature selection.

In [18]:
# Dummy-code explanatory predictors and calculate VIF.
# This cell is defensive: it can still return an empty VIF table gracefully
# instead of crashing when the explanatory sample is thin.

if explanatory_df.empty:
    print("The explanatory dataset is empty. Check the earlier explanatory data-prep/filtering cell.")
    y_expl = pd.Series(dtype=float)
    X_expl = pd.DataFrame()
    X_expl_const = pd.DataFrame()
    vif_df = pd.DataFrame(columns=["feature", "vif"])
    display(vif_df)
else:
    y_expl = explanatory_df["next_month_wellbeing_change"].copy()
    X_expl = explanatory_df.drop(columns=["next_month_wellbeing_change"]).copy()

    # Dummy code the categorical controls for OLS.
    X_expl = pd.get_dummies(X_expl, drop_first=True)
    X_expl = X_expl.replace([np.inf, -np.inf], np.nan)

    # Coerce everything to numeric to avoid statsmodels dtype issues.
    for col in X_expl.columns:
        X_expl[col] = pd.to_numeric(X_expl[col], errors="coerce")

    # Median-fill remaining numeric gaps for stability.
    X_expl = X_expl.fillna(X_expl.median(numeric_only=True))

    # Drop constant columns because they break VIF and add no explanatory value.
    X_expl = X_expl.loc[:, X_expl.nunique(dropna=False) > 1].copy()

    if X_expl.shape[1] == 0:
        print("No usable explanatory features remained after numeric conversion and constant-column filtering.")
        X_expl_const = pd.DataFrame()
        vif_df = pd.DataFrame(columns=["feature", "vif"])
        display(vif_df)
    else:
        X_expl_const = sm.add_constant(X_expl, has_constant="add")

        vif_rows = []
        for i, col in enumerate(X_expl_const.columns):
            if col == "const":
                continue

            try:
                vif_value = variance_inflation_factor(X_expl_const.values, i)
                vif_rows.append({"feature": col, "vif": vif_value})
            except Exception as e:
                vif_rows.append({"feature": col, "vif": np.nan})
                print(f"Skipping VIF issue for {col}: {e}")

        vif_df = pd.DataFrame(vif_rows)

        if vif_df.empty:
            print("VIF table is empty. No valid explanatory predictors were available.")
            vif_df = pd.DataFrame(columns=["feature", "vif"])
        else:
            vif_df = vif_df.sort_values("vif", ascending=False).reset_index(drop=True)

        display(vif_df.head(25))

Skipping VIF issue for wellbeing_composite: ufunc 'isfinite' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''
Skipping VIF issue for plans_created: ufunc 'isfinite' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''
Skipping VIF issue for achieved_plan_share: ufunc 'isfinite' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''
Skipping VIF issue for incident_count: ufunc 'isfinite' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''
Skipping VIF issue for high_severity_incidents: ufunc 'isfinite' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''
Sk

,feature,vif
0,wellbeing_composite,NaN
1,plans_created,NaN
2,achieved_plan_share,NaN
3,incident_count,NaN
4,high_severity_incidents,NaN
5,process_session_count,NaN
6,avg_session_duration,NaN
7,resident_age_years,NaN
8,resident_month_index,NaN
9,month_num,NaN


### Fit the explanatory model

Robust standard errors are used because the goal is a defensible relationship summary, not just raw coefficient output.

In [19]:
# Fit the explanatory OLS model with HC3 robust standard errors.
# Robust standard errors are used because the goal is a defensible relationship summary,
# not just raw coefficient output.

print("y_expl shape:", y_expl.shape if "y_expl" in globals() else "missing")
print("X_expl_const shape:", X_expl_const.shape if "X_expl_const" in globals() else "missing")

if "y_expl" not in globals() or "X_expl_const" not in globals():
    raise NameError("Run the explanatory data-prep cells before fitting the OLS model.")

# Make defensive copies so the original objects are not modified unexpectedly.
y_expl_model = pd.to_numeric(pd.Series(y_expl).copy(), errors="coerce")
X_expl_model = X_expl_const.copy()

# Convert boolean columns to integers for statsmodels.
bool_cols = X_expl_model.select_dtypes(include=["bool"]).columns.tolist()
for col in bool_cols:
    X_expl_model[col] = X_expl_model[col].astype(int)

# Convert any remaining object/category columns to numeric if possible.
# If conversion fails, they become NaN and will be removed below.
for col in X_expl_model.columns:
    if X_expl_model[col].dtype == "object" or str(X_expl_model[col].dtype).startswith("category"):
        X_expl_model[col] = pd.to_numeric(X_expl_model[col], errors="coerce")

# Replace inf values, then drop rows with any missing model inputs.
X_expl_model = X_expl_model.replace([np.inf, -np.inf], np.nan)
y_expl_model = y_expl_model.replace([np.inf, -np.inf], np.nan)

valid_mask = y_expl_model.notna() & X_expl_model.notna().all(axis=1)

print("Rows before numeric cleanup:", len(y_expl_model))
print("Rows after numeric cleanup:", int(valid_mask.sum()))
print("Dropped rows during cleanup:", int((~valid_mask).sum()))
print()

print("Column dtypes used in OLS:")
print(X_expl_model.dtypes)
print()

y_expl_model = y_expl_model.loc[valid_mask]
X_expl_model = X_expl_model.loc[valid_mask].copy()

if len(y_expl_model) == 0 or X_expl_model.shape[0] == 0:
    print("The explanatory modeling sample is empty after numeric cleanup.")
    expl_model = None
elif len(y_expl_model) != X_expl_model.shape[0]:
    print("The explanatory target and feature matrix do not have the same number of rows.")
    print("y_expl_model length:", len(y_expl_model))
    print("X_expl_model rows:", X_expl_model.shape[0])
    expl_model = None
else:
    expl_model = sm.OLS(y_expl_model.astype(float), X_expl_model.astype(float)).fit(cov_type="HC3")
    print(expl_model.summary())

# Keep cleaned versions available for later diagnostic/output cells.
y_expl_clean = y_expl_model
X_expl_const_clean = X_expl_model

y_expl shape: (474,)
X_expl_const shape: (474, 19)
Rows before numeric cleanup: 474
Rows after numeric cleanup: 474
Dropped rows during cleanup: 0

Column dtypes used in OLS:
const                        float64
wellbeing_composite          float64
plans_created                float64
achieved_plan_share          float64
incident_count               float64
high_severity_incidents      float64
process_session_count          int64
avg_session_duration         float64
resident_age_years           float64
resident_month_index         float64
month_num                      int32
case_category_Foundling        int64
case_category_Neglected        int64
case_category_Surrendered      int64
case_status_Closed             int64
case_status_Transferred        int64
region_Mindanao                int64
region_Visayas                 int64
region_unknown                 int64
dtype: object

                                 OLS Regression Results                                
Dep. Variable:     

### Save explanatory outputs and diagnostics

This makes the explanatory section more reusable and easier to discuss in the final project.

In [20]:
# Save explanatory coefficient and VIF outputs only if a model was fit.

if expl_model is None:
    print("No explanatory model was fit, so coefficient and diagnostic outputs were skipped.")
else:
    coef_df = pd.DataFrame(
        {
            "feature": expl_model.params.index,
            "coefficient": expl_model.params.values,
            "std_error": expl_model.bse.values,
            "t_value": expl_model.tvalues.values,
            "p_value": expl_model.pvalues.values,
        }
    ).sort_values("p_value", ascending=True)

    coef_output_path = OUTPUT_DIR / "intervention_effectiveness_explanatory_coefficients.csv"
    vif_output_path = OUTPUT_DIR / "intervention_effectiveness_explanatory_vif.csv"

    coef_df.to_csv(coef_output_path, index=False)
    vif_df.to_csv(vif_output_path, index=False)

    print("Saved coefficients to:", coef_output_path)
    print("Saved VIF table to:", vif_output_path)

    # Save two simple diagnostics for reporting.
    residuals = expl_model.resid
    fitted_vals = expl_model.fittedvalues

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(fitted_vals, residuals, alpha=0.6)
    ax.axhline(0, linestyle="--")
    ax.set_title("Explanatory Model Residuals vs Fitted")
    ax.set_xlabel("Fitted values")
    ax.set_ylabel("Residuals")
    resid_plot_path = OUTPUT_DIR / "intervention_effectiveness_explanatory_residuals_vs_fitted.png"
    fig.tight_layout()
    fig.savefig(resid_plot_path, dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(residuals, bins=30)
    ax.set_title("Explanatory Model Residual Distribution")
    ax.set_xlabel("Residual")
    ax.set_ylabel("Count")
    hist_plot_path = OUTPUT_DIR / "intervention_effectiveness_explanatory_residual_hist.png"
    fig.tight_layout()
    fig.savefig(hist_plot_path, dpi=150)
    plt.close(fig)

    print("Saved residual plot to:", resid_plot_path)
    print("Saved residual histogram to:", hist_plot_path)

Saved coefficients to: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs\intervention_effectiveness_explanatory_coefficients.csv
Saved VIF table to: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs\intervention_effectiveness_explanatory_vif.csv
Saved residual plot to: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs\intervention_effectiveness_explanatory_residuals_vs_fitted.png
Saved residual histogram to: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs\intervention_effectiveness_explanatory_residual_hist.png


## 4. Predictive Modeling & Safe Feature Selection

This section predicts:

`intervention_success_next_month`

The predictive workflow follows the stronger machine-learning path:

- time-aware holdout split
- grouped cross-validation by resident
- preprocessing inside pipelines
- safe feature selection inside the workflow
- model comparison

### Time-aware train and test split

The latest 20 percent of months become the held-out test period.

In [21]:
# Create train and test sets using a time-aware split.
# Prefer a calendar split when real month values exist.
# Fall back to a simple ordered split when the month field is sparse or missing.

if "metric_month" in panel_model.columns and panel_model["metric_month"].notna().sum() > 0:
    unique_months = sorted(panel_model["metric_month"].dropna().unique())
    test_month_count = max(1, math.ceil(len(unique_months) * 0.20))

    test_months = unique_months[-test_month_count:]
    train_months = unique_months[:-test_month_count]

    train_df = panel_model[panel_model["metric_month"].isin(train_months)].copy()
    test_df = panel_model[panel_model["metric_month"].isin(test_months)].copy()
else:
    train_df = pd.DataFrame()
    test_df = pd.DataFrame()

# Fall back to a simple ordered split if there are too few unique months
# or if the calendar-based split produced an empty side.
if train_df.empty or test_df.empty:
    order_cols = [col for col in ["resident_id", "resident_month_index", "metric_month"] if col in panel_model.columns]
    if not order_cols:
        order_cols = panel_model.columns.tolist()[:1]

    ordered_panel = panel_model.sort_values(order_cols).reset_index(drop=True)
    cutoff = int(len(ordered_panel) * 0.80)

    if cutoff <= 0:
        cutoff = max(1, len(ordered_panel) - 1)

    train_df = ordered_panel.iloc[:cutoff].copy()
    test_df = ordered_panel.iloc[cutoff:].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

if "metric_month" in train_df.columns and train_df["metric_month"].notna().sum() > 0:
    print("Train months:", train_df["metric_month"].min(), "to", train_df["metric_month"].max())
if "metric_month" in test_df.columns and test_df["metric_month"].notna().sum() > 0:
    print("Test months:", test_df["metric_month"].min(), "to", test_df["metric_month"].max())

Train shape: (18900, 67)
Test shape: (22, 67)
Train months: 2023-01-01 00:00:00 to 2026-04-01 00:00:00
Test months: 2026-05-01 00:00:00 to 2027-02-01 00:00:00


### Leakage control and feature definition

In [22]:
# Define target and drop leakage fields.
# This is where the predictive model matrix is created.

target_col = "intervention_success_next_month"

leakage_cols = [
    "next_metric_month",
    "next_month_wellbeing_composite",
    "next_month_incident_count",
    "next_month_high_severity_incidents",
    "next_month_wellbeing_change",
    "has_true_next_month",
    target_col,
]

identifier_cols = ["resident_id"]

drop_cols = [col for col in leakage_cols + identifier_cols if col in train_df.columns]

X_train = train_df.drop(columns=drop_cols, errors="ignore").copy()
X_test = test_df.drop(columns=drop_cols, errors="ignore").copy()

y_train = train_df[target_col].copy()
y_test = test_df[target_col].copy()

# Remove raw datetime columns because the model should not directly consume them.
datetime_cols = X_train.select_dtypes(include=["datetime64[ns]", "datetime64", "datetimetz"]).columns.tolist()
X_train = X_train.drop(columns=datetime_cols, errors="ignore")
X_test = X_test.drop(columns=datetime_cols, errors="ignore")

# Keep the same feature set on both sides of the split.
common_cols = sorted(set(X_train.columns).intersection(set(X_test.columns)))
X_train = X_train[common_cols].copy()
X_test = X_test[common_cols].copy()

# Keep a group vector for resident-level grouped CV.f.index, dtype=object
groups_train = train_df["resident_id"].fillna(-1).copy()

print("Dropped datetime columns:", datetime_cols)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
display(X_train.head())

Dropped datetime columns: ['metric_month', 'open_date']
X_train shape: (18900, 57)
X_test shape: (22, 57)


,achieved_plan_share,achieved_plans,achieved_plans_lag1,achieved_plans_lag2,active_plans,active_plans_lag1,active_plans_lag2,avg_session_duration,avg_session_duration_lag1,avg_session_duration_lag2,case_category,case_status,education_plan_share,education_plans,education_plans_lag1,education_plans_lag2,energy_level_score,energy_level_score_lag1,energy_level_score_lag2,health_plan_share,health_plans,health_plans_lag1,health_plans_lag2,health_record_count,high_severity_incidents,high_severity_incidents_lag1,high_severity_incidents_lag2,incident_count,incident_count_lag1,incident_count_lag2,incident_count_roll3_mean,month_num,plans_created,plans_created_lag1,plans_created_lag2,process_session_count,process_session_count_lag1,process_session_count_lag2,process_session_count_roll3_mean,region,resident_age_years,resident_month_index,safehouse_age_days,safehouse_id,safehouse_name,safety_plan_share,safety_plans,safety_plans_lag1,safety_plans_lag2,sleep_quality_score,sleep_quality_score_lag1,sleep_quality_score_lag2,wellbeing_composite,wellbeing_composite_lag1,wellbeing_composite_lag2,wellbeing_composite_roll3_mean,year_num
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,70.250000,NaN,NaN,Neglected,Active,0.0,0.0,0.0,0.0,2.9,NaN,NaN,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10,3.0,0.0,0.0,4,0.0,0.0,0.000000,NaN,17.6,1.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,3.18,NaN,NaN,3.04,NaN,NaN,NaN,2023
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,62.166667,70.250000,NaN,Neglected,Active,0.0,0.0,0.0,0.0,2.9,2.9,NaN,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10,3.0,3.0,0.0,6,4.0,0.0,4.000000,NaN,17.6,2.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,3.18,3.18,NaN,3.04,3.04,NaN,3.04,2023
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,105.000000,62.166667,70.250000,Neglected,Active,0.0,0.0,0.0,0.0,2.9,2.9,2.9,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10,3.0,3.0,3.0,1,6.0,4.0,5.000000,NaN,17.6,3.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,3.18,3.18,3.18,3.04,3.04,3.04,3.04,2023
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,63.000000,105.000000,62.166667,Neglected,Active,0.0,0.0,0.0,0.0,2.9,2.9,2.9,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10,3.0,3.0,3.0,3,1.0,6.0,3.666667,NaN,17.6,4.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,3.18,3.18,3.18,3.04,3.04,3.04,3.04,2023
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,79.000000,63.000000,105.000000,Neglected,Active,0.0,0.0,0.0,0.0,2.9,2.9,2.9,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10,3.0,3.0,3.0,2,3.0,1.0,3.333333,NaN,17.6,5.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,3.18,3.18,3.18,3.04,3.04,3.04,3.04,2023


### Feature typing and preprocessing

In [23]:
# Identify numeric and categorical features after cleanup.

numeric_features = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

# Treat safehouse_id as categorical if it remains in the feature matrix.
if "safehouse_id" in X_train.columns and "safehouse_id" in numeric_features:
    numeric_features.remove("safehouse_id")
    categorical_features.append("safehouse_id")

categorical_features = sorted(set(categorical_features))

print("Numeric feature count:", len(numeric_features))
print("Categorical feature count:", len(categorical_features))
print("Categorical features:", categorical_features)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

Numeric feature count: 52
Categorical feature count: 5
Categorical features: ['case_category', 'case_status', 'region', 'safehouse_id', 'safehouse_name']


### Candidate predictive models

A simpler baseline and a stronger nonlinear model are compared.

The logistic pipeline includes an embedded feature-selection step so feature selection stays inside the workflow instead of being done on the full dataset beforehand.

In [24]:
# Define candidate models.

feature_selector = SelectFromModel(
    estimator=LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=2000,
        random_state=SEED,
    ),
    threshold="median",
)

logit_pipeline = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("select", feature_selector),
        ("model", LogisticRegression(
            max_iter=3000,
            random_state=SEED,
            class_weight="balanced",
        )),
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=4,
            random_state=SEED,
            class_weight="balanced_subsample",
            n_jobs=-1,
        )),
    ]
)

MODELS = {
    "logistic_with_selection": logit_pipeline,
    "random_forest": rf_pipeline,
}

### Grouped cross-validation

In [25]:
# Grouped cross-validation on the training data.
# Use GroupKFold when repeated resident groups exist.
# Fall back to StratifiedKFold when grouped CV is not possible.

def evaluate_group_cv(model, X, y, groups, n_splits=5, random_state=42):
    groups_series = pd.Series(groups).reset_index(drop=True)
    y_series = pd.Series(y).reset_index(drop=True)

    # Count usable unique groups.
    n_unique_groups = groups_series.dropna().nunique()
    fold_results = []

    # Use grouped CV only when enough groups exist.
    if n_unique_groups >= 2:
        actual_splits = min(n_splits, int(n_unique_groups))
        cv = GroupKFold(n_splits=actual_splits)

        for train_idx, val_idx in cv.split(X, y_series, groups_series):
            X_fold_train = X.iloc[train_idx]
            X_fold_val = X.iloc[val_idx]
            y_fold_train = y_series.iloc[train_idx]
            y_fold_val = y_series.iloc[val_idx]

            model.fit(X_fold_train, y_fold_train)
            y_pred = model.predict(X_fold_val)

            if hasattr(model, "predict_proba"):
                y_score = model.predict_proba(X_fold_val)[:, 1]
            else:
                y_score = y_pred

            fold_results.append(
                {
                    "cv_type": "grouped",
                    "accuracy": accuracy_score(y_fold_val, y_pred),
                    "precision": precision_score(y_fold_val, y_pred, zero_division=0),
                    "recall": recall_score(y_fold_val, y_pred, zero_division=0),
                    "f1": f1_score(y_fold_val, y_pred, zero_division=0),
                    "roc_auc": roc_auc_score(y_fold_val, y_score) if y_fold_val.nunique() > 1 else np.nan,
                    "avg_precision": average_precision_score(y_fold_val, y_score) if y_fold_val.nunique() > 1 else np.nan,
                }
            )

    else:
        # Fall back to stratified CV when grouped CV is not possible.
        min_class_count = y_series.value_counts().min() if y_series.nunique() > 1 else 0
        actual_splits = min(n_splits, int(min_class_count)) if min_class_count >= 2 else 2

        cv = StratifiedKFold(n_splits=actual_splits, shuffle=True, random_state=random_state)

        for train_idx, val_idx in cv.split(X, y_series):
            X_fold_train = X.iloc[train_idx]
            X_fold_val = X.iloc[val_idx]
            y_fold_train = y_series.iloc[train_idx]
            y_fold_val = y_series.iloc[val_idx]

            model.fit(X_fold_train, y_fold_train)
            y_pred = model.predict(X_fold_val)

            if hasattr(model, "predict_proba"):
                y_score = model.predict_proba(X_fold_val)[:, 1]
            else:
                y_score = y_pred

            fold_results.append(
                {
                    "cv_type": "stratified_fallback",
                    "accuracy": accuracy_score(y_fold_val, y_pred),
                    "precision": precision_score(y_fold_val, y_pred, zero_division=0),
                    "recall": recall_score(y_fold_val, y_pred, zero_division=0),
                    "f1": f1_score(y_fold_val, y_pred, zero_division=0),
                    "roc_auc": roc_auc_score(y_fold_val, y_score) if y_fold_val.nunique() > 1 else np.nan,
                    "avg_precision": average_precision_score(y_fold_val, y_score) if y_fold_val.nunique() > 1 else np.nan,
                }
            )

    return fold_results

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("groups_train length:", len(groups_train))
print("Unique non-null groups:", pd.Series(groups_train).dropna().nunique())
print("Sample groups:", pd.Series(groups_train).dropna().head(10).tolist())
print("Class balance:")
print(pd.Series(y_train).value_counts(dropna=False))
print()

cv_rows = []

for model_name, model in MODELS.items():
    print(f"Running CV for: {model_name}")

    try:
        fold_rows = evaluate_group_cv(model, X_train, y_train, groups_train, n_splits=5)

        print(f"  Folds returned: {len(fold_rows)}")

        for row in fold_rows:
            row["model_name"] = model_name
            cv_rows.append(row)

    except Exception as e:
        print(f"  Failed: {e}")

cv_results = pd.DataFrame(cv_rows)

print()
print("Total CV rows collected:", len(cv_rows))

if cv_results.empty:
    print("Cross-validation produced no results.")
    cv_summary = pd.DataFrame(
        columns=[
            "model_name",
            "cv_type",
            "accuracy_mean", "accuracy_std",
            "precision_mean", "precision_std",
            "recall_mean", "recall_std",
            "f1_mean", "f1_std",
            "roc_auc_mean", "roc_auc_std",
            "avg_precision_mean", "avg_precision_std",
        ]
    )
else:
    display(cv_results)

    cv_summary = (
        cv_results
        .groupby(["model_name", "cv_type"], as_index=False)
        .agg(
            {
                "accuracy": ["mean", "std"],
                "precision": ["mean", "std"],
                "recall": ["mean", "std"],
                "f1": ["mean", "std"],
                "roc_auc": ["mean", "std"],
                "avg_precision": ["mean", "std"],
            }
        )
    )

    cv_summary.columns = [
        "_".join([piece for piece in col if piece]).strip("_")
        for col in cv_summary.columns.to_flat_index()
    ]

    display(cv_summary.sort_values(["f1_mean", "roc_auc_mean"], ascending=False))

X_train shape: (18900, 57)
y_train shape: (18900,)
groups_train length: 18900
Unique non-null groups: 61
Sample groups: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Class balance:
intervention_success_next_month
0    18583
1      317
Name: count, dtype: int64

Running CV for: logistic_with_selection
  Folds returned: 5
Running CV for: random_forest
  Folds returned: 5

Total CV rows collected: 10


,cv_type,accuracy,precision,recall,f1,roc_auc,avg_precision,model_name
0,grouped,0.712523,0.041556,0.810345,0.079058,0.823353,0.058416,logistic_with_selection
1,grouped,0.597998,0.037342,0.921875,0.071776,0.812441,0.060326,logistic_with_selection
2,grouped,0.651919,0.034457,0.730159,0.065808,0.744969,0.042445,logistic_with_selection
3,grouped,0.631718,0.035739,0.772727,0.068319,0.727987,0.042076,logistic_with_selection
4,grouped,0.657727,0.038491,0.772727,0.073329,0.754554,0.047605,logistic_with_selection
5,grouped,0.973484,0.328000,0.706897,0.448087,0.968583,0.469747,random_forest
6,grouped,0.973920,0.355372,0.671875,0.464865,0.943684,0.457808,random_forest
7,grouped,0.968817,0.307143,0.682540,0.423645,0.956675,0.412427,random_forest
8,grouped,0.970612,0.330827,0.666667,0.442211,0.956134,0.376402,random_forest
9,grouped,0.970526,0.335766,0.696970,0.453202,0.929382,0.449931,random_forest


,model_name,cv_type,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,avg_precision_mean,avg_precision_std
1,random_forest,grouped,0.971472,0.002164,0.331422,0.017277,0.684990,0.016861,0.446402,0.015210,0.950891,0.014904,0.433263,0.038336
0,logistic_with_selection,grouped,0.650377,0.041864,0.037517,0.002730,0.801567,0.072996,0.071658,0.005073,0.772661,0.042552,0.050174,0.008702


### Select the final predictive model and evaluate it on the held-out test set

In [26]:
# Run cross-validation across candidate models and summarize results.

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train length:", len(y_train))
print("y_test length:", len(y_test))
print("y_train value counts:")
print(pd.Series(y_train).value_counts(dropna=False))
print("Unique non-null groups in groups_train:", pd.Series(groups_train).dropna().nunique())
print()

cv_rows = []

for model_name, model in MODELS.items():
    print(f"Running CV for: {model_name}")

    try:
        fold_rows = evaluate_group_cv(model, X_train, y_train, groups_train, n_splits=5)
        print(f"  Fold rows returned: {len(fold_rows)}")

        for row in fold_rows:
            row["model_name"] = model_name
            cv_rows.append(row)

    except Exception as e:
        print(f"  Failed for {model_name}: {e}")

cv_results = pd.DataFrame(cv_rows)

if cv_results.empty:
    print("Cross-validation produced no usable rows.")
    cv_summary = pd.DataFrame(
        columns=[
            "model_name",
            "cv_type",
            "accuracy_mean", "accuracy_std",
            "precision_mean", "precision_std",
            "recall_mean", "recall_std",
            "f1_mean", "f1_std",
            "roc_auc_mean", "roc_auc_std",
            "avg_precision_mean", "avg_precision_std",
        ]
    )
else:
    display(cv_results)

    cv_summary = (
        cv_results
        .groupby(["model_name", "cv_type"], as_index=False)
        .agg(
            {
                "accuracy": ["mean", "std"],
                "precision": ["mean", "std"],
                "recall": ["mean", "std"],
                "f1": ["mean", "std"],
                "roc_auc": ["mean", "std"],
                "avg_precision": ["mean", "std"],
            }
        )
    )

    cv_summary.columns = [
        "_".join([piece for piece in col if piece]).strip("_")
        for col in cv_summary.columns.to_flat_index()
    ]

    display(cv_summary.sort_values(["f1_mean", "roc_auc_mean"], ascending=False))

X_train shape: (18900, 57)
X_test shape: (22, 57)
y_train length: 18900
y_test length: 22
y_train value counts:
intervention_success_next_month
0    18583
1      317
Name: count, dtype: int64
Unique non-null groups in groups_train: 61

Running CV for: logistic_with_selection
  Fold rows returned: 5
Running CV for: random_forest
  Fold rows returned: 5


,cv_type,accuracy,precision,recall,f1,roc_auc,avg_precision,model_name
0,grouped,0.712523,0.041556,0.810345,0.079058,0.823353,0.058416,logistic_with_selection
1,grouped,0.597998,0.037342,0.921875,0.071776,0.812441,0.060326,logistic_with_selection
2,grouped,0.651919,0.034457,0.730159,0.065808,0.744969,0.042445,logistic_with_selection
3,grouped,0.631718,0.035739,0.772727,0.068319,0.727987,0.042076,logistic_with_selection
4,grouped,0.657727,0.038491,0.772727,0.073329,0.754554,0.047605,logistic_with_selection
5,grouped,0.973484,0.328000,0.706897,0.448087,0.968583,0.469747,random_forest
6,grouped,0.973920,0.355372,0.671875,0.464865,0.943684,0.457808,random_forest
7,grouped,0.968817,0.307143,0.682540,0.423645,0.956675,0.412427,random_forest
8,grouped,0.970612,0.330827,0.666667,0.442211,0.956134,0.376402,random_forest
9,grouped,0.970526,0.335766,0.696970,0.453202,0.929382,0.449931,random_forest


,model_name,cv_type,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,avg_precision_mean,avg_precision_std
1,random_forest,grouped,0.971472,0.002164,0.331422,0.017277,0.684990,0.016861,0.446402,0.015210,0.950891,0.014904,0.433263,0.038336
0,logistic_with_selection,grouped,0.650377,0.041864,0.037517,0.002730,0.801567,0.072996,0.071658,0.005073,0.772661,0.042552,0.050174,0.008702


## 5. Evaluation Outputs, Feature Importance, and Deployment Files

This is the section meant to improve the deployment consistency of the earlier notebooks.

In [27]:
# Save cross-validation and test outputs.
# Only save predictive outputs after the final model selection succeeded.

if "cv_results" not in globals() or "cv_summary" not in globals() or "test_metrics" not in globals():
    print("Predictive outputs were not saved because one or more upstream predictive objects are missing.")
else:
    cv_output_path = OUTPUT_DIR / "intervention_effectiveness_cv_results.csv"
    cv_summary_path = OUTPUT_DIR / "intervention_effectiveness_cv_summary.csv"
    test_metrics_path = OUTPUT_DIR / "intervention_effectiveness_test_metrics.csv"

    cv_results.to_csv(cv_output_path, index=False)
    cv_summary.to_csv(cv_summary_path, index=False)
    test_metrics.to_csv(test_metrics_path, index=False)

    print("Saved CV results to:", cv_output_path)
    print("Saved CV summary to:", cv_summary_path)
    print("Saved test metrics to:", test_metrics_path)

Predictive outputs were not saved because one or more upstream predictive objects are missing.


In [28]:
# Run cross-validation across candidate models and summarize results.
# This version prints model-level failures so the exact CV problem is visible.

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train length:", len(y_train))
print("y_test length:", len(y_test))
print("y_train value counts:")
print(pd.Series(y_train).value_counts(dropna=False))
print("Unique non-null groups in groups_train:", pd.Series(groups_train).dropna().nunique())
print()

cv_rows = []

for model_name, model in MODELS.items():
    print(f"Running CV for: {model_name}")

    try:
        fold_rows = evaluate_group_cv(model, X_train, y_train, groups_train, n_splits=5)
        print(f"  Fold rows returned: {len(fold_rows)}")

        for row in fold_rows:
            row["model_name"] = model_name
            cv_rows.append(row)

    except Exception as e:
        print(f"  Failed for {model_name}: {repr(e)}")

cv_results = pd.DataFrame(cv_rows)

print()
print("Total CV rows collected:", len(cv_rows))

if cv_results.empty:
    print("Cross-validation produced no usable rows.")
    cv_summary = pd.DataFrame(
        columns=[
            "model_name",
            "cv_type",
            "accuracy_mean", "accuracy_std",
            "precision_mean", "precision_std",
            "recall_mean", "recall_std",
            "f1_mean", "f1_std",
            "roc_auc_mean", "roc_auc_std",
            "avg_precision_mean", "avg_precision_std",
        ]
    )
else:
    display(cv_results)

    cv_summary = (
        cv_results
        .groupby(["model_name", "cv_type"], as_index=False)
        .agg(
            {
                "accuracy": ["mean", "std"],
                "precision": ["mean", "std"],
                "recall": ["mean", "std"],
                "f1": ["mean", "std"],
                "roc_auc": ["mean", "std"],
                "avg_precision": ["mean", "std"],
            }
        )
    )

    cv_summary.columns = [
        "_".join([piece for piece in col if piece]).strip("_")
        for col in cv_summary.columns.to_flat_index()
    ]

    display(cv_summary.sort_values(["f1_mean", "roc_auc_mean"], ascending=False))

X_train shape: (18900, 57)
X_test shape: (22, 57)
y_train length: 18900
y_test length: 22
y_train value counts:
intervention_success_next_month
0    18583
1      317
Name: count, dtype: int64
Unique non-null groups in groups_train: 61

Running CV for: logistic_with_selection
  Fold rows returned: 5
Running CV for: random_forest
  Fold rows returned: 5

Total CV rows collected: 10


,cv_type,accuracy,precision,recall,f1,roc_auc,avg_precision,model_name
0,grouped,0.712523,0.041556,0.810345,0.079058,0.823353,0.058416,logistic_with_selection
1,grouped,0.597998,0.037342,0.921875,0.071776,0.812441,0.060326,logistic_with_selection
2,grouped,0.651919,0.034457,0.730159,0.065808,0.744969,0.042445,logistic_with_selection
3,grouped,0.631718,0.035739,0.772727,0.068319,0.727987,0.042076,logistic_with_selection
4,grouped,0.657727,0.038491,0.772727,0.073329,0.754554,0.047605,logistic_with_selection
5,grouped,0.973484,0.328000,0.706897,0.448087,0.968583,0.469747,random_forest
6,grouped,0.973920,0.355372,0.671875,0.464865,0.943684,0.457808,random_forest
7,grouped,0.968817,0.307143,0.682540,0.423645,0.956675,0.412427,random_forest
8,grouped,0.970612,0.330827,0.666667,0.442211,0.956134,0.376402,random_forest
9,grouped,0.970526,0.335766,0.696970,0.453202,0.929382,0.449931,random_forest


,model_name,cv_type,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,avg_precision_mean,avg_precision_std
1,random_forest,grouped,0.971472,0.002164,0.331422,0.017277,0.684990,0.016861,0.446402,0.015210,0.950891,0.014904,0.433263,0.038336
0,logistic_with_selection,grouped,0.650377,0.041864,0.037517,0.002730,0.801567,0.072996,0.071658,0.005073,0.772661,0.042552,0.050174,0.008702


In [29]:
print("cv_summary exists:", "cv_summary" in globals())
if "cv_summary" in globals():
    print("cv_summary shape:", cv_summary.shape)

print("best_model exists:", "best_model" in globals())

cv_summary exists: True
cv_summary shape: (2, 14)
best_model exists: False


In [31]:
print("cv_summary exists:", "cv_summary" in globals())
if "cv_summary" in globals():
    print("cv_summary shape:", cv_summary.shape)

print("best_model exists:", "best_model" in globals())

print("MODELS keys:", list(MODELS.keys()) if "MODELS" in globals() else "MODELS missing")

cv_summary exists: True
cv_summary shape: (2, 14)
best_model exists: False
MODELS keys: ['logistic_with_selection', 'random_forest']


In [32]:
# Select the best predictive model using mean F1, then fit on the full training data.

print("cv_summary exists:", "cv_summary" in globals())
if "cv_summary" in globals():
    print("cv_summary shape:", cv_summary.shape)
    display(cv_summary)

if "cv_summary" not in globals():
    raise NameError("cv_summary is not defined. Run the cross-validation summary cell first.")

if cv_summary.empty:
    raise ValueError("cv_summary is empty. Cross-validation did not produce any usable model results.")

best_model_name = cv_summary.sort_values(["f1_mean", "roc_auc_mean"], ascending=False).iloc[0]["model_name"]
print("Selected model name from cv_summary:", best_model_name)

if best_model_name not in MODELS:
    raise KeyError(f"{best_model_name} is not a key in MODELS. Available keys: {list(MODELS.keys())}")

best_model = clone(MODELS[best_model_name])
best_model.fit(X_train, y_train)

print("best_model created successfully:", "best_model" in globals())
print("Selected predictive model:", best_model_name)

test_pred = best_model.predict(X_test)
test_proba = best_model.predict_proba(X_test)[:, 1]

test_metrics = pd.DataFrame(
    [
        {
            "model_name": best_model_name,
            "accuracy": accuracy_score(y_test, test_pred),
            "precision": precision_score(y_test, test_pred, zero_division=0),
            "recall": recall_score(y_test, test_pred, zero_division=0),
            "f1": f1_score(y_test, test_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, test_proba) if pd.Series(y_test).nunique() > 1 else np.nan,
            "avg_precision": average_precision_score(y_test, test_proba),
        }
    ]
)

display(test_metrics)
print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))
print()
print("Classification report:")
print(classification_report(y_test, test_pred, zero_division=0))

cv_summary exists: True
cv_summary shape: (2, 14)


,model_name,cv_type,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,avg_precision_mean,avg_precision_std
0,logistic_with_selection,grouped,0.650377,0.041864,0.037517,0.002730,0.801567,0.072996,0.071658,0.005073,0.772661,0.042552,0.050174,0.008702
1,random_forest,grouped,0.971472,0.002164,0.331422,0.017277,0.684990,0.016861,0.446402,0.015210,0.950891,0.014904,0.433263,0.038336


Selected model name from cv_summary: random_forest
best_model created successfully: True
Selected predictive model: random_forest


,model_name,accuracy,precision,recall,f1,roc_auc,avg_precision
0,random_forest,1.0,0.0,0.0,0.0,NaN,0.0


Confusion matrix:
[[22]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        22

    accuracy                           1.00        22
   macro avg       1.00      1.00      1.00        22
weighted avg       1.00      1.00      1.00        22



In [33]:
print("best_model exists:", "best_model" in globals())

best_model exists: True


In [34]:
# Save model artifact and metadata for downstream app integration.
# This step is skipped gracefully if the final predictive model was never fit.

if "best_model" not in globals():
    print("Model artifact and metadata were skipped because best_model does not exist.")
else:
    model_output_path = OUTPUT_DIR / "intervention_effectiveness_model.joblib"
    metadata_output_path = OUTPUT_DIR / "intervention_effectiveness_metadata.json"
    scores_output_path = OUTPUT_DIR / "intervention_effectiveness_scores.csv"

    priority_output = test_df.copy()
    priority_output["pressure_probability"] = test_proba
    priority_output["predicted_intervention_success"] = test_pred
    if target_col in priority_output.columns:
        priority_output["actual_intervention_success"] = priority_output[target_col]

    priority_output = priority_output.sort_values("pressure_probability", ascending=False).reset_index(drop=True)
    priority_output.to_csv(scores_output_path, index=False)

    joblib.dump(best_model, model_output_path)

    metadata = {
        "pipeline_name": "intervention_effectiveness",
        "selected_model_name": best_model_name,
        "random_seed": SEED,
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "feature_count": int(len(common_cols)),
        "target_name": target_col,
        "output_files": {
            "panel": str(panel_output_path),
            "cv_results": str(cv_output_path) if "cv_output_path" in globals() else None,
            "cv_summary": str(cv_summary_path) if "cv_summary_path" in globals() else None,
            "test_metrics": str(test_metrics_path) if "test_metrics_path" in globals() else None,
            "feature_importance": str(importance_output_path) if "importance_output_path" in globals() else None,
            "scores": str(scores_output_path) if "scores_output_path" in globals() else None,
            "explanatory_coefficients": str(coef_output_path) if "coef_output_path" in globals() else None,
            "explanatory_vif": str(vif_output_path) if "vif_output_path" in globals() else None,
            "model_artifact": str(model_output_path),
        },
    }

    with open(metadata_output_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print("Saved scored output to:", scores_output_path)
    print("Saved model artifact to:", model_output_path)
    print("Saved metadata to:", metadata_output_path)

Saved model artifact to: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs\intervention_effectiveness_model.joblib
Saved metadata to: c:\Users\Ashns\OneDrive\INTEX26\INTEX_W2026\ml-pipelines\generated_outputs\intervention_effectiveness_metadata.json


## 6. Final Integration Notes

### How this pipeline can be used in the app

This notebook creates outputs that fit naturally into the INTEX site:

- `intervention_effectiveness_scores.csv` can populate a staff-facing priority list
- `intervention_effectiveness_feature_importance.csv` can support analytics and reporting
- the explanatory coefficient and VIF files can support the presentation and written justification
- the saved model artifact and metadata improve repeatability and deployment readiness

### Suggested dashboard use

A staff dashboard could show:

- residents with the **lowest predicted intervention success probability**
- current-month intervention intensity by safehouse
- average next-month wellbeing change by intervention mix
- a short explanation card summarizing the strongest relationship signals from the explanatory model

## Notes for project integration

- This notebook is intentionally stricter about explanatory analysis than some of the earlier pipelines.
- It is also more explicit about deployment outputs and metadata.
- If your raw schema uses different field names, update the alias maps instead of rewriting the whole notebook.
- If your team wants this target to focus on **education change** or **reintegration change** instead of wellbeing change, the same notebook structure can be reused with a different outcome table.

In [ ]:
# Mirror outputs back to the local repo fallback after writing to Azure ML output storage when available.
import shutil
LOCAL_OUTPUT_DIR = PROJECT_ROOT / "generated_outputs"
if OUTPUT_DIR.resolve() != LOCAL_OUTPUT_DIR.resolve():
    LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for artifact in OUTPUT_DIR.glob('*'):
        if artifact.is_file():
            shutil.copy2(artifact, LOCAL_OUTPUT_DIR / artifact.name)
    print(f'Mirrored outputs to local fallback: {LOCAL_OUTPUT_DIR.resolve()}')
